In [1]:
!pip install crewai crewai_tools langchain_community sentence-transformers langchain-groq langchain_huggingface --quiet

In [2]:
!pip install langchain

In [1]:
import os
#os.environ['GROQ_API_KEY']='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'
os.environ['TAVILY_API_KEY']='tvly-z66T6o4dug8blrXv5eaZ4Ronsd3cVMZy___Q'


In [1]:
from langchain.agents import tool
from crewai_tools import PDFSearchTool
from crewai import Agent, Crew, Task,Process
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI


/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal-MW7Qkosx/lib/python3.10/site-packages/pydantic/_internal/_config.py:295: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal-MW7Qkosx/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal

In [2]:

"""
llm =ChatOpenAI(
    openai_api_key=os.environ['GROQ_API_KEY'],
    openai_api_base="https://api.groq.com/openai/v1",
    #model_name="llama3-8b-8192",
    model_name="llama-3.2-3b-preview",
    temperature=0.1,
    max_tokens=1000,

)
"""
# OR Use Ollama to run Locally
from crewai import Agent, Task, Process, Crew, LLM
ollama_llm = LLM(
            model="ollama/llama3.2",
            api_base="http://localhost:11434"
            )


In [3]:
#!pip install tf-keras

In [4]:
"""
rag_tool = PDFSearchTool(pdf='attentions_is_all_you_need.pdf',
    config=dict(
        llm=dict(
            provider="groq", # or google, openai, anthropic, llama2, ...
            config=dict(
                model="llama3-8b-8192",
                # temperature=0.5,
                # top_p=1,
                # stream=true,
            ),
        ),
        embedder=dict(
            provider="huggingface", # or openai, ollama, ...
            config=dict(
                model="BAAI/bge-small-en-v1.5",
                #task_type="retrieval_document",
                # title="Embeddings",
            ),
        ),
    )
)

"""

rag_tool = PDFSearchTool(pdf='attentions_is_all_you_need.pdf',
    config=dict(
        llm=dict(
            provider="ollama", # or google, openai, anthropic, llama2, ...
            config=dict(
                model="llama3.2:latest",
                base_url="http://localhost:11434"
            ),
        ),
        embedder=dict(
                provider="huggingface", # or openai, ollama, ...
                config=dict(
                    model="BAAI/bge-small-en-v1.5",
                    #task_type="retrieval_document",
                    # title="Embeddings",
                ),
            ),
    )
)


/opt/anaconda3/envs/dev_env/lib/python3.10/abc.py:123: ResourceWarning: unclosed <socket.socket fd=77, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 56013), raddr=('127.0.0.1', 11434)>
  return _abc_subclasscheck(cls, subclass)
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimodal-MW7Qkosx/lib/python3.10/site-packages/embedchain/embedder/huggingface.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=self.config.model, model_kwargs=self.config.model_kwargs)
/Users/tharhtet/.local/share/virtualenvs/Multi-Agent-Healthcare-iEESBots-Multimo

In [5]:
question = "How did self-attention mechanism evolve in large language models in pdf?"
rag_tool.run(question)

Using Tool: Search a PDF's content


'Relevant Content:\nmakes it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 22, 23, 19]. End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [28]. To the best of our knowledge, however, t

In [7]:


@tool
def simple_router_tool(question : str) -> str:
  """Routes queries to the appropriate resources"""

  print("question gg : ",question)
  if "pdf" in question.lower():
    return "vectorstore"
  else:
    return "websearch"



In [8]:
test_input = {"question": "What does the  say about machine learning in pdf?"}
simple_router_tool.run(test_input)

question gg :  What does the  say about machine learning in pdf?


'vectorstore'

In [9]:
import yaml

def config_reader(file_path):
    with open(file_path, 'r') as f:
        data = yaml.load(f, Loader=yaml.SafeLoader)
        
    # Print the values as a dictionary
    return data


In [10]:
from crewai.project import CrewBase, agent, crew, task

In [11]:


#Agent 1

agent1_config = "config/agent_1.yaml"

Router_Agent = Agent(
                config=config_reader(agent1_config)["agent_1"],

                tools = [simple_router_tool],
                allow_delegation=False,
                verbose=True,
                llm=ollama_llm,)
    



In [12]:
#Task 1
router_task = Task(
    description=("Analyse the keywords in the question {question}"
    "Based on the keywords decide whether it is eligible for a vectorstore search or a web search."
    "Return a single word 'vectorstore' if it is eligible for vectorstore search."
    "Return a single word 'websearch' if it is eligible for web search."
    "Do not provide any other premable or explaination."
    ),
    expected_output=("Give a binary choice 'websearch' or 'vectorstore' based on the question"
    "Do not provide any other premable or explaination."),
    agent=Router_Agent,
    
)

In [13]:
crew1 = Crew(
    agents = [Router_Agent],
    tasks = [router_task],
    verbose= True,
)

test_input =  {'question': {'description': "What does the  say about machine learning in pdf?", 'type': 'str'}}
#test_input = {"question": "What does the  say about machine learning in pdf?"}
result1 = crew1.kickoff(inputs=test_input)

# Agent: Router
## Task: Analyse the keywords in the question {'description': 'What does the  say about machine learning in pdf?', 'type': 'str'}Based on the keywords decide whether it is eligible for a vectorstore search or a web search.Return a single word 'vectorstore' if it is eligible for vectorstore search.Return a single word 'websearch' if it is eligible for web search.Do not provide any other premable or explaination.
question gg :  What does the  say about machine learning in pdf?


# Agent: Router
## Using tool: simple_router_tool
## Tool Input: 
"{\"question\": \"What does the  say about machine learning in pdf?\"}"
## Tool Output: 
vectorstore


# Agent: Router
## Final Answer: 
vectorstore




In [12]:
print(result1)

vectorstore
